# Вычисляемые поля

На том же `CreditSample.txt` строим производные признаки: дату обработки, сумму в у.е., булевы флаги по условиям, степенное преобразование срока и сегментацию заёмщиков по правилам (аналог функции IFF в Deductor)

In [1]:
#Импорт библиотек
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
#Импорт файла
df = pd.read_csv("data/CreditSample.txt", sep="\t", encoding="cp1251")
df.head(5)

,Код,"Размер ссуды, руб","Срок ссуды, мес",Цель ссуды,"Среднемесячный доход, руб","Среднемесячный расход, руб",Основное направление расходов,Количество лет,Семейное положение,Количество иждивенцев,Наличие недвижимости,Давать кредит
0,1,8000,18,Покупка и ремонт недвижимости,6000,3000,Затраты на образование (в т.ч. детей),38,Да,3,Нет,ИСТИНА
1,2,7500,24,Иное,14000,12000,Покупка товаров длит. пользования,61,Да,0,Нет,ЛОЖЬ
2,3,17000,12,Покупка автомобиля,3500,1500,"Содержание/аренда недвижимости, а/т",45,Нет,1,Да,ЛОЖЬ
3,4,12500,36,Оплата за образование,5500,4500,Затраты на образование (в т.ч. детей),34,Нет,1,Да,ИСТИНА
4,5,16500,6,Иное,7500,4000,"Содержание/аренда недвижимости, а/т",55,Нет,3,Нет,ИСТИНА


In [3]:
#Новое поле "Дата обработки" (текущая дата)
df["Дата обработки"] = datetime.today().strftime("%d/%m/%Y")
df["Дата обработки"]

0      24/09/2026
1      24/09/2026
2      24/09/2026
3      24/09/2026
4      24/09/2026
          ...    
195    24/09/2026
196    24/09/2026
197    24/09/2026
198    24/09/2026
199    24/09/2026
Name: Дата обработки, Length: 200, dtype: str

In [4]:
#Новое поле "Размер ссуды у.е." = Размер ссуды / 30 (с округлением до 2 знаков)
df["Размер ссуды у.е."] = (df["Размер ссуды, руб"] / 30).round(2)
df["Размер ссуды у.е."]

0      266.67
1      250.00
2      566.67
3      416.67
4      550.00
        ...  
195    950.00
196    683.33
197    233.33
198    233.33
199    233.33
Name: Размер ссуды у.е., Length: 200, dtype: float64

In [5]:
#Новое поле "Флаг"
#Условие: Среднемесячный доход > 2000 и Наличие недвижимости = Да
df["Флаг"] = (df["Среднемесячный доход, руб"] > 2000) & (df["Наличие недвижимости"] == "Да")
df["Флаг"]

0      False
1      False
2       True
3       True
4      False
       ...  
195     True
196    False
197    False
198    False
199    False
Name: Флаг, Length: 200, dtype: bool

In [6]:
#Новое поле "Флаг_Кредит"
#Значение 1, если Флаг = TRUE и Давать кредит = FALSE
df["Флаг_Кредит"] = np.where((df["Флаг"] == True) & (df["Давать кредит"] == "ЛОЖЬ"), 1, 0)
df["Флаг_Кредит"]

0      0
1      0
2      1
3      0
4      0
      ..
195    0
196    0
197    0
198    0
199    0
Name: Флаг_Кредит, Length: 200, dtype: int64

In [7]:
#Новое поле RATE = (Срок ссуды)^0.6
df["RATE"] = df["Срок ссуды, мес"] ** 0.6
df["RATE"]

0      5.664525
1      6.731731
2      4.441286
3      8.585814
4      2.930156
         ...   
195    5.664525
196    6.731731
197    4.441286
198    2.930156
199    8.585814
Name: RATE, Length: 200, dtype: float64

In [8]:
#Новое поле "Сегмент" (по правилам IFF)
def assign_segment(row):
    age = row["Количество лет"]
    income = row["Среднемесячный доход, руб"]
    expense = row["Среднемесячный расход, руб"]

    if age >= 50 and income < 6000:
        return "Сегмент 1"
    elif age < 30 and expense >= 5500:
        return "Сегмент 2"
    else:
        return "Сегмент 3"

df["Сегмент"] = df.apply(assign_segment, axis=1)
df.head(5)

,Код,"Размер ссуды, руб","Срок ссуды, мес",Цель ссуды,"Среднемесячный доход, руб","Среднемесячный расход, руб",Основное направление расходов,Количество лет,Семейное положение,Количество иждивенцев,Наличие недвижимости,Давать кредит,Дата обработки,Размер ссуды у.е.,Флаг,Флаг_Кредит,RATE,Сегмент
0,1,8000,18,Покупка и ремонт недвижимости,6000,3000,Затраты на образование (в т.ч. детей),38,Да,3,Нет,ИСТИНА,24/09/2026,266.67,False,0,5.664525,Сегмент 3
1,2,7500,24,Иное,14000,12000,Покупка товаров длит. пользования,61,Да,0,Нет,ЛОЖЬ,24/09/2026,250.00,False,0,6.731731,Сегмент 3
2,3,17000,12,Покупка автомобиля,3500,1500,"Содержание/аренда недвижимости, а/т",45,Нет,1,Да,ЛОЖЬ,24/09/2026,566.67,True,1,4.441286,Сегмент 3
3,4,12500,36,Оплата за образование,5500,4500,Затраты на образование (в т.ч. детей),34,Нет,1,Да,ИСТИНА,24/09/2026,416.67,True,0,8.585814,Сегмент 3
4,5,16500,6,Иное,7500,4000,"Содержание/аренда недвижимости, а/т",55,Нет,3,Нет,ИСТИНА,24/09/2026,550.00,False,0,2.930156,Сегмент 3


In [9]:
#Сохранение
df.to_csv("data/Project3_final.csv", index=False, encoding="utf-8-sig")